# 08 — Évaluation : baseline et agent, sur le gold set

**Objectif** : comparer plusieurs configurations sur les mêmes questions, avec des chiffres.

1. **Chargement** : connexions, gold set.
2. **Configurations** : définies en dur, une par dictionnaire. C'est la seule cellule à modifier pour tester autre chose.
3. **Exécution** : chaque configuration répond à toutes les questions. Les résultats sont **enregistrés sur disque** : relancer le notebook ne rappelle pas l'API pour une configuration déjà évaluée.
4. **Métriques de retrieval** (gratuites) : hit@k, recall@k, MRR.
5. **Métriques de génération** (RAGAS, juge Claude Haiku) : faithfulness, answer relevancy, context precision, context recall.
6. **Abstention, citations, coût, latence.**
7. **Graphiques** (plotly), puis **examen des échecs**.


> **Coût** : chaque configuration fait une génération par question (plus les tours de l'agent), puis RAGAS appelle Haiku une dizaine de fois par question. Pour une première passe, limiter le nombre de questions avec `N_MAX` (section 2).
>
> **Variabilité** : Claude Sonnet 5 n'accepte pas de température fixe. Deux exécutions de la même configuration peuvent donc différer un peu.

## 1. Connexions et gold set

On réutilise le moteur de recherche et le générateur du package, ce sont exactement ceux de l'application.

In [1]:
import asyncio
import hashlib
import json
import time
from pathlib import Path

import anthropic
import numpy as np
import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv
from psycopg.rows import dict_row

from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.config import ParamsGeneration, charger_config
from assistant_regles.rag.embeddings import EncodeurBGEM3
from assistant_regles.rag.generation import Generateur, charger_prompt, titre_resultat
from assistant_regles.rag.recherche import COLONNES_RESULTAT, MoteurRecherche, Resultat
from assistant_regles.rag.store import TABLE

RACINE = trouver_racine()
load_dotenv(RACINE / ".env")
config_rag = charger_config()

conn = psycopg.connect(autocommit=True)
moteur = MoteurRecherche(EncodeurBGEM3.charger(config_rag.embeddings), conn)
client = anthropic.Anthropic(max_retries=8)
PHRASE_ABSTENTION = config_rag.generation.phrase_abstention

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Colonnes attendues dans `gold_set.xlsx` :

| colonne | contenu |
|---|---|
| `id` | identifiant de la question (q001…) |
| `question` | la question telle qu'un joueur la poserait |
| `references` | liste JSON de références, ex. `[{"code": "24.27"}]`, `[{"page": 12}]`, `[{"code": "15.11", "page": 57}]` ; `[]` ou vide = hors corpus |
| `reponse_attendue` | réponse de référence, avec tes mots (sert à RAGAS) |
| `categorie` | simple, paraphrase, vocabulaire_joueur, regles_proches, multi_sections, hors_corpus… |

Une référence correspond à un passage si **tous ses critères renseignés** sont vérifiés : même code, et/ou page comprise entre `page_debut` et `page_fin`.

In [2]:
gold = pd.read_excel(RACINE / "data/core-rules/gold_set_w40k_v11.xlsx", dtype={"id": str})
print("colonnes :", list(gold.columns))

colonnes : ['id', 'question', 'references', 'reponse_attendue', 'categorie', 'theme']


In [3]:
def lire_references(valeur):
    """Cellule 'references' -> liste de dicts {code?, page?}. Vide = hors corpus."""
    if pd.isna(valeur) or not str(valeur).strip():
        return []
    refs = json.loads(str(valeur))
    for ref in refs:
        if not (ref.get("code") or ref.get("page") is not None):
            raise ValueError(f"référence vide : {valeur}")
    return refs


gold["refs"] = gold["references"].map(lire_references)
gold["hors_corpus"] = gold["refs"].map(len) == 0
gold["reponse_attendue"] = gold.get("reponse_attendue", pd.Series("", index=gold.index)).fillna("")
gold["categorie"] = gold.get("categorie", pd.Series("", index=gold.index)).fillna("")

print(len(gold), "questions, dont", int(gold["hors_corpus"].sum()), "hors corpus")
gold["categorie"].value_counts()

108 questions, dont 8 hors corpus


categorie
difficile    54
simple       28
version      26
Name: count, dtype: int64

## 2. Configurations à comparer

**C'est la cellule à modifier.** Chaque dictionnaire est une configuration :

- `type` : `"baseline"` (question brute -> top-k -> générateur) ou `"agent"` (agent de recherche -> sélection -> générateur) ;
- **générateur** (les deux types) : `modele`, `effort`, `max_tokens`, `granularite`, `prompt` (fichier dans `rag/prompts/`) ;
- **baseline** : `k` ;
- **agent** : `modele_agent`, `effort_agent` (`None` pour ne pas l'envoyer), `max_tours`, `max_passages`, `k_recherche`, `amorce`, `prompt_agent` (texte, défini dans la cellule suivante).

Le fichier de résultats de chaque configuration porte son **nom et une empreinte** de tous ses paramètres (texte des prompts compris). Si tu modifies un paramètre, la configuration est recalculée ; sinon, les résultats enregistrés sont réutilisés.

In [4]:
PROMPT_AGENT_V1 = """Tu es l'agent de recherche d'un assistant d'arbitrage pour le jeu Warhammer 40,000 (11e édition).
Tu ne réponds pas au joueur : un autre modèle rédigera la réponse, uniquement à partir des passages que tu retiendras.
Ton travail est de trouver dans le livre de règles de base tous les passages nécessaires pour répondre complètement
et correctement à la question, et seulement ceux-là.

Méthode :
1. Identifie la ou les règles en jeu. Les joueurs n'emploient pas toujours le vocabulaire du livre : traduis leurs
   termes en termes probables du livre (noms de phases et d'actions, mots-clés en MAJUSCULES).
2. Cherche avec rechercher_regles, une requête par règle concernée. Si les résultats ne sont pas pertinents,
   reformule autrement. Effectue plusieurs recherches : lance une première recherche, analyse les résultats, puis effectue d'autres recherches.
   Tu dois effectuer au moins une recherche. Les cas sans recherchent sont rares et ne correpondent qu'a des questions très simples.
3. Suis avec lire_regle les renvois utiles indiqués dans les passages.
4. Pense aux exceptions et aux cas particuliers : une règle générale a souvent des restrictions ailleurs dans le livre.
5. Quand tu as tout ce qu'il faut, appelle retenir_passages avec les étiquettes des passages utiles, du plus au moins
   important, au maximum {max_passages}. N'utilise que des étiquettes que tu as vues.

Les questions hors sujet ne doivent pas recevoir de réponse.Si aucun passage du livre ne concerne la question, il s'agit probablement d'une question hors sujet : appelle retenir_passages avec une liste vide.
Appuies-toi uniquement sur les informations contenues dans la base de données desrègles.
Tu disposes d'au plus {max_tours_recherche} tours de recherche : sois efficace, regroupe les appels indépendants
dans un même tour.
"""

GENERATEUR = {"modele": "claude-sonnet-5", 
              "effort": "medium", 
              "max_tokens": 4096,
              "granularite": "paragraphe", 
              "prompt": "systeme_vf.md"}

AGENT = {"max_tours": 6, 
         "max_passages": 8,
           "k_recherche": 5, 
           "amorce": True, 
           "prompt_agent": PROMPT_AGENT_V1}

CONFIGS = [
    {"nom": "baseline_k5", "type": "baseline", "k": 5, **GENERATEUR},
    {"nom": "baseline_k8", "type": "baseline", "k": 8, **GENERATEUR},
    {"nom": "agent_haiku", "type": "agent", "modele_agent": "claude-haiku-4-5-20251001", "effort_agent": None,
     **AGENT, **GENERATEUR},
    {"nom": "agent_sonnet_high", "type": "agent", "modele_agent": "claude-sonnet-5", "effort_agent": "high",
     **AGENT, **GENERATEUR},
    {"nom": "agent_opus_high", "type": "agent", "modele_agent": "claude-opus-5-5", "effort_agent": "high",
     **AGENT, **GENERATEUR},
]

N_MAX = None             # ex. 5 pour un premier essai rapide ; None = tout le gold set
DOSSIER = RACINE / "data/eval/resultats"
DOSSIER.mkdir(parents=True, exist_ok=True)

TARIFS = {  # $ par million de tokens (entrée, sortie) : à vérifier sur la page de tarifs d'Anthropic
    "claude-haiku-4-5-20251001": (1, 5),
    "claude-sonnet-5": (2, 10),
    "claude-opus-5-5": (4, 20),
}
MAX_TOKENS_AGENT = 16_000

## 3. Exécution

Deux fonctions, une par type de configuration. Chacune renvoie, pour une question :

- `contextes` : les passages transmis au générateur, **dans l'ordre** (top-k pour la baseline, ordre d'importance choisi par l'agent) ;
- la réponse, ses citations, l'abstention ;
- tokens, coût et durée (recherche comprise).

L'agent est celui du notebook 07, recopié ici en plus court et piloté par la configuration. Il rejoindra le package (`rag/agent.py`) une fois ses réglages choisis grâce à cette évaluation.

In [5]:
def generateur_pour(cfg):
    """Le générateur de l'application, avec les paramètres de la configuration."""
    params = ParamsGeneration(modele=cfg["modele"], effort=cfg["effort"], max_tokens=cfg["max_tokens"],
                              granularite=cfg["granularite"], prompt_systeme=cfg["prompt"],
                              phrase_abstention=PHRASE_ABSTENTION)
    return Generateur(client, params, charger_prompt(cfg["prompt"], PHRASE_ABSTENTION))


def cout(modele, tokens_entree, tokens_sortie):
    prix_entree, prix_sortie = TARIFS.get(modele, (0, 0))
    return (tokens_entree * prix_entree + tokens_sortie * prix_sortie) / 1e6


def en_contexte(r):
    """Ce qu'on garde d'un passage pour les métriques (et pour RAGAS)."""
    return {"code": r.code, "page_debut": r.page_debut, "page_fin": r.page_fin, "texte": r.texte}

In [6]:
# ---------- Agent de recherche (notebook 07, version courte) ----------
SQL_LIRE = f"SELECT {', '.join(COLONNES_RESULTAT)} FROM {TABLE} WHERE code = %s ORDER BY page_debut, partie"


def lire_regle(code):
    with conn.cursor(row_factory=dict_row) as cur:
        lignes = cur.execute(SQL_LIRE, (code,)).fetchall()
    return [Resultat(rang=i, score=float("nan"), **l) for i, l in enumerate(lignes, start=1)]


def outils_agent(max_passages):
    return [
        {"name": "rechercher_regles",
         "description": ("Recherche dans le livre de règles de base de Warhammer 40,000 (11e édition) les passages les "
                         "plus proches, par le sens, de la requête. Écris la requête avec le vocabulaire probable du livre "
                         "(noms de phases, d'actions, mots-clés en majuscules). Une requête par règle. Chaque passage porte "
                         "une étiquette (P1, P2…), son code, ses pages et ses renvois."),
         "input_schema": {"type": "object", "required": ["requete"], "properties": {
             "requete": {"type": "string"}, "k": {"type": "integer", "minimum": 1, "maximum": 10}}}},
        {"name": "lire_regle",
         "description": "Renvoie le texte complet d'une règle à partir de son code « XX.YY » (pour suivre un renvoi).",
         "input_schema": {"type": "object", "required": ["code"], "properties": {
             "code": {"type": "string", "pattern": r"^\d{2}\.\d{2}$"}}}},
        {"name": "retenir_passages",
         "description": ("Termine la recherche : étiquettes des passages nécessaires pour répondre, du plus au moins "
                         "important. Liste vide si aucun passage ne concerne la question."),
         "input_schema": {"type": "object", "required": ["etiquettes", "justification"], "properties": {
             "etiquettes": {"type": "array", "items": {"type": "string", "pattern": r"^P\d+$"}, "maxItems": max_passages},
             "justification": {"type": "string"}}}},
    ]


def chercher_agent(question, cfg):
    """Renvoie (passages retenus, infos sur la recherche)."""
    etiquettes, par_id = {}, {}
    infos = {"tours": 0, "appels": 0, "fin": "", "tokens_entree": 0, "tokens_sortie": 0, "requetes": []}
    prompt = cfg["prompt_agent"].format(max_passages=cfg["max_passages"], max_tours_recherche=cfg["max_tours"] - 1)

    def etiqueter(resultats):
        blocs, deja = [], []
        for r in resultats:
            if r.id in par_id:
                deja.append(par_id[r.id])
                continue
            e = f"P{len(etiquettes) + 1}"
            etiquettes[e], par_id[r.id] = r, e
            renvois = f" — renvois : {', '.join(r.codes_cites)}" if r.codes_cites else ""
            blocs.append({"type": "text", "text": f"[{e}] {titre_resultat(r)}{renvois}\n{r.texte}"})
        if deja:
            blocs.append({"type": "text", "text": f"Déjà fournis plus haut : {', '.join(deja)}."})
        return blocs or [{"type": "text", "text": "Aucun passage trouvé."}]

    contenu = []
    if cfg["amorce"]:
        contenu += [{"type": "text", "text": "Passages trouvés par une première recherche avec la question brute :"}]
        contenu += etiqueter(moteur.rechercher(question, cfg["k_recherche"]))
    contenu.append({"type": "text", "text": f"Question du joueur : {question}"})
    messages = [{"role": "user", "content": contenu}]
    options = {"output_config": {"effort": cfg["effort_agent"]}} if cfg["effort_agent"] else {}

    for tour in range(1, cfg["max_tours"] + 1):
        reponse = client.messages.create(model=cfg["modele_agent"], max_tokens=MAX_TOKENS_AGENT, system=prompt,
                                         tools=outils_agent(cfg["max_passages"]), messages=messages, **options)
        infos["tours"] = tour
        infos["tokens_entree"] += reponse.usage.input_tokens
        infos["tokens_sortie"] += reponse.usage.output_tokens
        messages.append({"role": "assistant", "content": reponse.content})
        appels = [b for b in reponse.content if b.type == "tool_use"]

        fin = next((b for b in appels if b.name == "retenir_passages"), None)
        if fin is not None:
            demandees = list(dict.fromkeys(fin.input.get("etiquettes", [])))
            infos["fin"] = "retenue"
            return [etiquettes[e] for e in demandees if e in etiquettes][:cfg["max_passages"]], infos
        if not appels:
            infos["fin"] = "repli_texte"
            break
        if tour == cfg["max_tours"]:
            infos["fin"] = "repli_budget"
            break

        resultats_outils = []
        for b in appels:
            infos["appels"] += 1
            try:
                if b.name == "rechercher_regles":
                    infos["requetes"].append(b.input["requete"])
                    trouves = moteur.rechercher(b.input["requete"], int(b.input.get("k", cfg["k_recherche"])))
                else:
                    infos["requetes"].append(f"lire_regle {b.input['code']}")
                    trouves = lire_regle(b.input["code"])
                resultats_outils.append({"type": "tool_result", "tool_use_id": b.id, "content": etiqueter(trouves)})
            except Exception as e:
                resultats_outils.append({"type": "tool_result", "tool_use_id": b.id,
                                         "content": f"Erreur : {e}", "is_error": True})
        if tour == cfg["max_tours"] - 1:
            resultats_outils.append({"type": "text", "text":
                "Budget de recherche épuisé : appelle maintenant retenir_passages avec les passages utiles."})
        messages.append({"role": "user", "content": resultats_outils})

    return list(etiquettes.values())[:cfg["max_passages"]], infos   # repli : passages vus, dans l'ordre

In [7]:
def repondre(question, cfg, generateur):
    """Une question, une configuration -> une ligne de résultats."""
    debut = time.perf_counter()
    ligne = {"agent_tours": None, "agent_appels": None, "agent_fin": None, "agent_requetes": None}
    cout_recherche = 0.0

    if cfg["type"] == "baseline":
        passages = moteur.rechercher(question, cfg["k"])
    else:
        passages, infos = chercher_agent(question, cfg)
        cout_recherche = cout(cfg["modele_agent"], infos["tokens_entree"], infos["tokens_sortie"])
        ligne.update(agent_tours=infos["tours"], agent_appels=infos["appels"], agent_fin=infos["fin"],
                     agent_requetes=infos["requetes"])

    if passages:
        rep = generateur.generer(question, passages)
        ligne.update(reponse=rep.texte, abstention=rep.abstention,
                     cites=sorted({c.rang_resultat for c in rep.citations}),   # rangs dans `contextes` (à partir de 1)
                     part_citee=rep.part_citee,
                     cout_generation=cout(cfg["modele"], rep.tokens_entree, rep.tokens_sortie))
    else:   # l'agent n'a rien retenu : abstention sans appel au générateur
        ligne.update(reponse=PHRASE_ABSTENTION, abstention=True, cites=[], part_citee=0.0, cout_generation=0.0)

    ligne.update(contextes=[en_contexte(r) for r in passages], cout_recherche=cout_recherche,
                 cout=cout_recherche + ligne["cout_generation"], duree=time.perf_counter() - debut)
    return ligne


def fichier_resultats(cfg):
    """Nom lisible + empreinte de tous les paramètres, texte du prompt de génération compris."""
    prompt = charger_prompt(cfg["prompt"], PHRASE_ABSTENTION)
    empreinte = hashlib.sha256(json.dumps({**cfg, "_prompt": prompt.empreinte}, sort_keys=True).encode()).hexdigest()
    return DOSSIER / f"{cfg['nom']}_{empreinte[:8]}.jsonl"


def lire_fichier(chemin):
    return pd.read_json(chemin, lines=True, dtype={"id": str}) if chemin.exists() else pd.DataFrame({"id": []})


def evaluer(cfg, questions):
    """Exécute une configuration sur les questions pas encore traitées, puis renvoie toutes ses lignes.

    Les résultats sont ajoutés au fichier de la configuration : passer de N_MAX=5 au gold set complet
    ne recalcule que les questions manquantes, et une exécution interrompue reprend où elle s'est arrêtée.
    """
    chemin = fichier_resultats(cfg)
    deja = lire_fichier(chemin)
    if "erreur" in deja:
        deja = deja[deja["erreur"].isna()]   # les questions en erreur sont recalculées
    a_faire = questions[~questions["id"].isin(deja["id"])]
    print(f"{cfg['nom']:>20} : {len(questions) - len(a_faire)} déjà calculée(s), {len(a_faire)} à calculer")

    if len(a_faire):
        generateur = generateur_pour(cfg)
        lignes = []
        for i, q in enumerate(a_faire.itertuples(), start=1):
            try:
                ligne = repondre(q.question, cfg, generateur)
            except Exception as e:   # une question en erreur ne fait pas perdre les autres
                ligne = {"erreur": f"{type(e).__name__} : {e}", "contextes": [], "cites": [], "abstention": None,
                         "cout": 0.0, "duree": None}
            lignes.append({"config": cfg["nom"], "id": q.id, **ligne})
            print(f"\r{cfg['nom']:>20} : {i}/{len(a_faire)}", end="")
        print()
        deja = pd.concat([deja, pd.DataFrame(lignes)], ignore_index=True)
        deja.to_json(chemin, orient="records", lines=True, force_ascii=False)
    return deja[deja["id"].isin(questions["id"])]

In [8]:
questions = gold if N_MAX is None else gold.head(N_MAX)
brut = pd.concat([evaluer(cfg, questions) for cfg in CONFIGS], ignore_index=True)
res = brut.merge(gold[["id", "question", "refs", "hors_corpus", "reponse_attendue", "categorie"]], on="id")

if "erreur" not in res:
    res["erreur"] = None
erreurs = res[res["erreur"].notna()]
print(len(res), "lignes |", len(erreurs), "erreur(s)")
print(erreurs.groupby("config").size().to_string() if len(erreurs) else "aucune erreur")
res = res[res["erreur"].isna()].copy()   # les métriques ne portent que sur les questions sans erreur

         baseline_k5 : 108 déjà calculée(s), 0 à calculer
         baseline_k8 : 47 déjà calculée(s), 61 à calculer
         baseline_k8 : 61/61
         agent_haiku : 0 déjà calculée(s), 108 à calculer
         agent_haiku : 108/108
   agent_sonnet_high : 0 déjà calculée(s), 108 à calculer
   agent_sonnet_high : 108/108
     agent_opus_high : 0 déjà calculée(s), 108 à calculer
     agent_opus_high : 108/108
540 lignes | 117 erreur(s)
config
agent_opus_high      108
agent_sonnet_high      9


In [9]:
print(erreurs.groupby("config").size())
erreurs["erreur"].str[:150].value_counts().head(10)

config
agent_opus_high      108
agent_sonnet_high      9
dtype: int64


erreur
BadRequestError : Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'You have reached your specified API usage    117
Name: count, dtype: int64

## 4. Métriques de retrieval

Calculées sur les `contextes` transmis au générateur, pour les questions **dans le corpus** uniquement :

- **hit@k** : au moins une référence est retrouvée dans les k premiers passages ;
- **recall@k** : part des références retrouvées dans les k premiers passages (pour les questions multi-règles) ;
- **MRR** : moyenne de 1 / rang du premier passage pertinent (1 = parfait, 0 = jamais trouvé) ;
- **@contexte** : sur tout le contexte transmis, quelle que soit sa taille. C'est la mesure qui compte pour le générateur. Attention : une configuration qui transmet plus de passages est avantagée ; compare-la avec la colonne `nb_passages`.

In [29]:
K_LISTE = [1, 3, 5]


def correspond(contexte, ref):
    if ref.get("code") and contexte["code"] != ref["code"]:
        return False
    return not (ref.get("page") is not None and not contexte["page_debut"] <= ref["page"] <= contexte["page_fin"])


def metriques_retrieval(ligne):
    contextes, refs = ligne["contextes"], ligne["refs"]
    if not refs:
        return {}
    # rang (à partir de 1) du premier passage qui correspond à chaque référence, ou None
    rangs = [next((i for i, c in enumerate(contextes, start=1) if correspond(c, ref)), None) for ref in refs]
    trouves = [r for r in rangs if r is not None]
    m = {"mrr": 1 / min(trouves) if trouves else 0.0,
         "hit@contexte": float(bool(trouves)),
         "recall@contexte": len(trouves) / len(refs)}
    for k in K_LISTE:
        m[f"hit@{k}"] = float(any(r <= k for r in trouves))
        m[f"recall@{k}"] = sum(r <= k for r in trouves) / len(refs)
    # la réponse cite-t-elle un passage pertinent ?
    cites = [contextes[i - 1] for i in ligne["cites"] if i <= len(contextes)]
    m["ref_citee"] = float(any(correspond(c, ref) for c in cites for ref in refs))
    return m


In [ ]:
res.head()

,id,config,agent_tours,agent_appels,agent_fin,agent_requetes,reponse,abstention,cites,part_citee,...,faithfulness,answer_relevancy,context_precision,context_recall,erreur,question,refs,hors_corpus,reponse_attendue,categorie
0,q001,baseline_k5,NaN,NaN,NaN,NaN,Ni l'un ni l'autre exactement : on part du **b...,False,"[1, 2]",0.677626,...,0.727273,0.695609,1.0,0.750000,NaN,"Pour mesurer la distance entre deux figurines,...","[{'code': '01.04'}, {'code': '17.02'}]",False,On peut mesurer les distances à tout moment. S...,simple
1,q002,baseline_k5,NaN,NaN,NaN,NaN,"D'après les extraits, une unité qui échoue son...",False,"[1, 2]",0.373057,...,1.000000,0.605953,0.0,0.500000,NaN,Mon unité vient de rater son test d’ébranlemen...,"[{'code': '01.07'}, {'code': '08.03'}, {'code'...",False,Tant qu’elle est ébranlée (01.07) : la CO de t...,difficile
2,q003,baseline_k5,NaN,NaN,NaN,NaN,"Oui, vous devez faire un jet d'ébranlement pou...",False,"[1, 2, 3]",0.631579,...,0.909091,0.652318,1.0,0.600000,NaN,"J’avais une escouade de 10 figurines, il en re...","[{'code': '08.03'}, {'page': 31}, {'page': 86}]",False,"Oui. À l’étape Ébranlement (08.03), le joueur ...",version
3,q004,baseline_k5,NaN,NaN,NaN,NaN,"Non, ce n'est pas automatique : il faut réussi...",False,"[3, 4]",0.490017,...,0.944444,0.712020,1.0,0.833333,NaN,Mon unité a été ébranlée au tour dernier. Est-...,"[{'code': '08.03'}, {'code': '01.07'}, {'page'...",False,"Non. En V11, une unité ébranlée le reste jusqu...",version
4,q005,baseline_k5,NaN,NaN,NaN,NaN,Le joueur adverse (celui dont l'unité tire) de...,False,"[1, 3, 4]",0.421801,...,0.400000,0.716809,1.0,0.800000,NaN,"Pendant mon tour, mon adversaire utilise Tir e...","[{'code': '01.03'}, {'code': '15.08'}]",False,Chaque fois qu’une unité est choisie pour tire...,difficile


In [32]:
metriques = pd.DataFrame(list(res.apply(metriques_retrieval, axis=1)), index=res.index)
res = res.drop(columns=metriques.columns, errors="ignore").join(metriques)   # rejouable sans doublons
res["nb_passages"] = res["contextes"].map(len)

COLONNES_RETRIEVAL = [f"hit@{k}" for k in K_LISTE] + ["hit@contexte", "recall@contexte", "mrr", "ref_citee"]
dans_corpus = res[~res["hors_corpus"]]
tableau_retrieval = dans_corpus.groupby("config", sort=False)[COLONNES_RETRIEVAL + ["nb_passages"]].mean().round(3)
tableau_retrieval

,hit@1,hit@3,hit@5,hit@contexte,recall@contexte,mrr,ref_citee,nb_passages
config,,,,,,,,
baseline_k5,0.63,0.89,0.94,0.94,0.728,0.757,0.93,5.00
baseline_k8,0.32,0.42,0.45,0.45,0.350,0.369,0.45,3.76
agent_haiku,0.05,0.05,0.05,0.05,0.036,0.050,0.05,0.11
agent_sonnet_high,0.05,0.05,0.05,0.05,0.042,0.050,0.05,0.13
agent_opus_high,0.05,0.05,0.05,0.05,0.048,0.050,0.05,0.21


## 5. Métriques de génération (RAGAS, juge Claude Haiku)

Les quatre métriques de la spec, sur les questions **dans le corpus** :

- **faithfulness** : part des affirmations de la réponse étayées par le contexte. C'est la métrique qui détecte une réponse « correcte mais non ancrée », que Claude aurait tirée de sa mémoire ;
- **answer relevancy** : la réponse répond-elle bien à la question ? (RAGAS génère des questions à partir de la réponse et les compare à l'originale avec BGE-M3) ;
- **context precision** : les passages utiles sont-ils en tête du contexte ? (nécessite `reponse_attendue`) ;
- **context recall** : le contexte contient-il tout ce qu'il faut pour la réponse attendue ? (nécessite `reponse_attendue`).

Les scores sont ajoutés aux fichiers de résultats : un score déjà calculé n'est jamais recalculé (seuls les manquants, ou ceux en erreur, le sont).

Deux réglages techniques :

- RAGAS envoie par défaut `temperature` **et** `top_p`. Je retire `top_p` : les modèles Claude récents refusent a priori le cumul des deux (à vérifier si Haiku 4.5 lève une erreur) ;
- les appels au juge sont faits en parallèle (8 à la fois) pour ne pas attendre des heures.

In [33]:
from anthropic import AsyncAnthropic
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
)

juge = llm_factory("claude-haiku-4-5-20251001", provider="anthropic", client=AsyncAnthropic(), max_tokens=4096)
juge.model_args.pop("top_p", None)
juge.model_args.pop("temperature", None)
embeddings_juge = HuggingFaceEmbeddings(model="BAAI/bge-m3")

METRIQUES_RAGAS = {
    "faithfulness": Faithfulness(llm=juge),
    "answer_relevancy": AnswerRelevancy(llm=juge, embeddings=embeddings_juge),
    "context_precision": ContextPrecisionWithReference(llm=juge),
    "context_recall": ContextRecall(llm=juge),
}
PARALLELISME = 8

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [35]:
async def score_ragas(nom, metrique, ligne):
    """Un score RAGAS pour une ligne ; None si non applicable ou en erreur."""
    textes = [c["texte"] for c in ligne["contextes"]]
    if ligne["hors_corpus"] or not textes or not isinstance(ligne.get("reponse"), str):
        return None
    try:
        if nom == "faithfulness":
            r = await metrique.ascore(user_input=ligne["question"], response=ligne["reponse"], retrieved_contexts=textes)
        elif nom == "answer_relevancy":
            r = await metrique.ascore(user_input=ligne["question"], response=ligne["reponse"])
        elif not ligne["reponse_attendue"]:
            return None   # context precision / recall : réponse attendue nécessaire
        elif nom == "context_precision":
            r = await metrique.ascore(user_input=ligne["question"], reference=ligne["reponse_attendue"],
                                      retrieved_contexts=textes)
        else:
            r = await metrique.ascore(user_input=ligne["question"], retrieved_contexts=textes,
                                      reference=ligne["reponse_attendue"])
        return r.value
    except Exception as e:
        print(f"  [{nom}] {ligne['config']}/{ligne['id']} : {type(e).__name__} : {e}")
        return None


async def evaluer_ragas(resultats):
    """Calcule les scores manquants (NaN), en parallèle limité."""
    limite = asyncio.Semaphore(PARALLELISME)
    taches, cibles = [], []
    for nom in METRIQUES_RAGAS:
        if nom not in resultats:
            resultats[nom] = np.nan
    for i, ligne in resultats.iterrows():
        for nom, metrique in METRIQUES_RAGAS.items():
            if pd.isna(resultats.at[i, nom]):
                async def tache(nom=nom, metrique=metrique, ligne=ligne):
                    async with limite:
                        return await score_ragas(nom, metrique, ligne)
                taches.append(tache())
                cibles.append((i, nom))
    valeurs = await asyncio.gather(*taches)
    for (i, nom), valeur in zip(cibles, valeurs):
        resultats.loc[i, nom] = valeur
    calcules = sum(v is not None for v in valeurs)
    print(f"{calcules} score(s) calculé(s) ; {len(valeurs) - calcules} non applicable(s) ou en erreur (laissés vides)")
    return resultats


res = await evaluer_ragas(res)

# Enregistrement des scores dans le fichier de chaque configuration (pour ne pas les recalculer)
COLONNES_RAGAS = list(METRIQUES_RAGAS)
for cfg in CONFIGS:
    chemin = fichier_resultats(cfg)
    scores = res.loc[res["config"] == cfg["nom"], ["id"] + COLONNES_RAGAS]
    fichier = lire_fichier(chemin).drop(columns=COLONNES_RAGAS, errors="ignore").merge(scores, on="id", how="left")
    fichier.to_json(chemin, orient="records", lines=True, force_ascii=False)

tableau_ragas = res[~res["hors_corpus"]].groupby("config", sort=False)[list(METRIQUES_RAGAS)].mean().round(3)
tableau_ragas

548 score(s) calculé(s) ; 1512 non applicable(s) ou en erreur (laissés vides)


,faithfulness,answer_relevancy,context_precision,context_recall
config,,,,
baseline_k5,0.784,0.610,0.709,0.731
baseline_k8,0.790,0.618,0.763,0.768
agent_haiku,0.786,0.666,0.900,0.743
agent_sonnet_high,0.830,0.684,0.883,0.697
agent_opus_high,0.732,0.692,0.890,0.738


## 6. Abstention, coût et latence

- **abstention juste** : sur les questions hors corpus, part des réponses qui s'abstiennent (idéal : 1) ;
- **fausse abstention** : sur les questions du corpus, part des réponses qui s'abstiennent à tort (idéal : 0) ;
- **coût** et **durée** moyens par question, recherche de l'agent comprise ;
- pour l'agent : tours et appels d'outils moyens, et **replis** (l'agent n'a pas conclu par `retenir_passages`).

In [36]:
def resume_config(groupe):
    hors = groupe[groupe["hors_corpus"]]
    dedans = groupe[~groupe["hors_corpus"]]
    agent = groupe["agent_fin"].notna()
    return pd.Series({
        "abstention_juste": hors["abstention"].mean() if len(hors) else np.nan,
        "fausse_abstention": dedans["abstention"].mean(),
        "cout_moyen_centimes": groupe["cout"].mean() * 100,
        "duree_moyenne_s": groupe["duree"].mean(),
        "tours_agent": groupe.loc[agent, "agent_tours"].mean() if agent.any() else np.nan,
        "appels_agent": groupe.loc[agent, "agent_appels"].mean() if agent.any() else np.nan,
        "replis_agent": (groupe.loc[agent, "agent_fin"] != "retenue").mean() if agent.any() else np.nan,
    })


tableau_operationnel = pd.DataFrame({nom: resume_config(groupe) for nom, groupe in res.groupby("config", sort=False)}).T
tableau_operationnel.index.name = "config"
tableau_operationnel = tableau_operationnel.round(3)
tableau_operationnel

,abstention_juste,fausse_abstention,cout_moyen_centimes,duree_moyenne_s,tours_agent,appels_agent,replis_agent
config,,,,,,,
baseline_k5,1.0,0.060,1.581,7.627,NaN,NaN,NaN
baseline_k8,NaN,0.043,0.901,8.631,NaN,NaN,NaN
agent_haiku,NaN,0.000,0.132,14.013,3.0,3.4,0.0
agent_sonnet_high,NaN,0.000,0.113,9.788,1.2,0.2,0.0
agent_opus_high,NaN,0.000,0.355,13.835,2.2,1.6,0.0


## 7. Graphiques

In [37]:
# Retrieval : une barre par configuration et par métrique
donnees = tableau_retrieval[COLONNES_RETRIEVAL].reset_index().melt(id_vars="config", var_name="métrique",
                                                                    value_name="score")
fig = px.bar(donnees, x="métrique", y="score", color="config", barmode="group", range_y=[0, 1],
             title="Retrieval (questions du corpus)")
fig.show()

# Génération : les quatre métriques RAGAS
donnees = tableau_ragas.reset_index().melt(id_vars="config", var_name="métrique", value_name="score")
fig = px.bar(donnees, x="métrique", y="score", color="config", barmode="group", range_y=[0, 1],
             title="Génération (RAGAS, juge Claude Haiku)")
fig.show()

In [38]:
# Qualité contre coût : chaque point est une configuration, la taille donne la durée moyenne
synthese = tableau_retrieval.join(tableau_ragas).join(tableau_operationnel).reset_index()
fig = px.scatter(synthese, x="cout_moyen_centimes", y="hit@contexte", size="duree_moyenne_s", color="config",
                 hover_data=["mrr", "faithfulness", "duree_moyenne_s"], size_max=40,
                 title="Qualité du retrieval contre coût (taille = durée moyenne)",
                 labels={"cout_moyen_centimes": "coût moyen par question (centimes de $)"})
fig.update_yaxes(range=[0, 1.05])
fig.show()

# Par catégorie de question : où chaque configuration réussit ou échoue
par_categorie = dans_corpus.groupby(["categorie", "config"], sort=False)["hit@contexte"].mean().unstack()
fig = px.imshow(par_categorie, text_auto=".2f", color_continuous_scale="RdYlGn", zmin=0, zmax=1, aspect="auto",
                title="hit@contexte par catégorie de question")
fig.show()

## 8. Examen des échecs

Les chiffres disent **combien** ; cette section dit **lesquelles**. Pour une configuration : les questions du corpus où aucune référence n'est dans le contexte, avec les codes attendus et les codes fournis. Pour l'agent, ses requêtes montrent **pourquoi** il a raté.

In [39]:
CONFIG_EXAMINEE = "agent_sonnet_high"   # à changer

echecs = res[(res["config"] == CONFIG_EXAMINEE) & ~res["hors_corpus"] & (res["hit@contexte"] == 0)]
print(f"{len(echecs)} échec(s) de retrieval pour {CONFIG_EXAMINEE}\n")
def decrire_reference(ref):
    return " ".join(x for x in (ref.get("code"), f"p. {ref['page']}" if ref.get("page") is not None else None) if x)


def decrire_contexte(c):
    return f"{c['code'] or '(sans code)'} p. {c['page_debut']}"


for ligne in echecs.itertuples():
    print(f"[{ligne.id}] ({ligne.categorie}) {ligne.question}")
    print(f"   attendus : {[decrire_reference(r) for r in ligne.refs]}")
    print(f"   fournis  : {[decrire_contexte(c) for c in ligne.contextes]}")
    if isinstance(ligne.agent_requetes, list):
        print(f"   requêtes de l'agent : {ligne.agent_requetes}")
    print()

95 échec(s) de retrieval pour agent_sonnet_high

[q006] (difficile) J’ai un Capitaine attaché à 5 Intercessors. Combien de pertes avant que l’unité soit à demi-effectif ? Et si tous les Intercessors meurent, le Capitaine qui a tous ses PV est-il en dessous de son demi-effectif ?
   attendus : ['p. 86']
   fournis  : []

[q007] (difficile) Mon véhicule seul a 11 PV et il lui en reste 5 (ou 6). Est-il à demi-effectif pour le test d’ébranlement ?
   attendus : ['p. 86', 'p. 31', '08.03']
   fournis  : []

[q008] (simple) Combien de points de commandement je gagne à chaque round de bataille ?
   attendus : ['08.02', 'p. 30']
   fournis  : []

[q009] (version) Quelle est la règle de cohésion pour une grosse unité de 20 Boyz ?
   attendus : ['03.03']
   fournis  : []

[q010] (difficile) Après des pertes, mon unité se retrouve coupée en deux groupes et n’est plus en cohésion. Qu’est-ce qui se passe ?
   attendus : ['03.03']
   fournis  : []

[q011] (version) À quelle distance mes figurines so

In [43]:
ligne.reponse

"Le joueur adverse (celui qui contrôle l'unité en Tir en État d'Alerte) devient le joueur actif le temps de résoudre ce tir.\n\nEn effet, la règle générale précise : chaque fois qu'une unité est choisie pour tirer ou choisie pour combattre, le joueur en contrôle de cette unité est le joueur actif jusqu'à ce que ces attaques soient résolues.\n\nComme le Tir en État d'Alerte permet à une unité non engagée amie (de l'adversaire, dans ce cas) de tirer à la fin de votre phase de Mouvement, c'est bien votre adversaire — contrôleur de l'unité qui tire — qui devient temporairement le joueur actif pour la résolution de ce tir, même si c'est votre tour de jeu en cours."

In [40]:
# Faithfulness faible : réponses peut-être non ancrées, à relire
SEUIL = 0.7
suspectes = res[(res["config"] == CONFIG_EXAMINEE) & (res["faithfulness"] < SEUIL)]
for ligne in suspectes.itertuples():
    print(f"[{ligne.id}] faithfulness={ligne.faithfulness:.2f} | {ligne.question}\n{ligne.reponse}\n{'-' * 80}")

[q005] faithfulness=0.40 | Pendant mon tour, mon adversaire utilise Tir en État d’Alerte sur une de mes unités. Pendant ce tir, qui est le joueur actif ?
Le joueur adverse (celui qui contrôle l'unité en Tir en État d'Alerte) devient le joueur actif le temps de résoudre ce tir.

En effet, la règle générale précise : chaque fois qu'une unité est choisie pour tirer ou choisie pour combattre, le joueur en contrôle de cette unité est le joueur actif jusqu'à ce que ces attaques soient résolues.

Comme le Tir en État d'Alerte permet à une unité non engagée amie (de l'adversaire, dans ce cas) de tirer à la fin de votre phase de Mouvement, c'est bien votre adversaire — contrôleur de l'unité qui tire — qui devient temporairement le joueur actif pour la résolution de ce tir, même si c'est votre tour de jeu en cours.
--------------------------------------------------------------------------------


## Bilan

- **L'agent améliore-t-il le retrieval** (hit@contexte, recall, MRR), et sur quelles catégories ?
- **Au prix de quoi** : coût et durée par question ?
- **Quel modèle pour l'agent** : Haiku suffit-il, ou Sonnet/Opus apportent-ils un gain net ?
- **La génération suit-elle** : faithfulness et context recall meilleurs avec la sélection de l'agent ?
- **Abstention** : tenue sur le hors corpus, sans fausses abstentions ?

Les chiffres retenus deviendront la **baseline officielle** du README, et les réglages de l'agent ceux de `rag/agent.py`.

In [ ]:
conn.close()